# **Artificial Neural Networks and Deep Learning**

---

## **Lecture 1: Feedforward Neural Networks**

<img src="https://drive.google.com/uc?export=view&id=1Wi6qDtuvd18q7EIGOzdHHOYxquAiwNcp" width="300"/>


### 🌐 **Google Drive Connection**

In [ ]:
from google.colab import drive
drive.mount("/gdrive")
current_dir = "/gdrive/My\\ Drive/[2026-2027]\\ AN2DL/Lecture\\ 1"
%cd $current_dir

### ⚙️ **Libraries Import**

In [ ]:
# Install the libraries that are not preinstalled on Colab
!pip install -q torchinfo torchview

# Set seed for reproducibility
SEED = 42

import os
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import clear_output
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from torchinfo import summary
from torchview import draw_graph

# Trained weights are saved in this folder, one file per model
MODELS_DIR = 'models/Feedforward Neural Networks'
os.makedirs(MODELS_DIR, exist_ok=True)


def weights_path(name):
    """File where the weights of the model called `name` are saved."""
    return f"{MODELS_DIR}/{re.sub(r'[^A-Za-z0-9]+', '_', name).strip('_').lower()}.pt"


# Seed the Python, NumPy and PyTorch random number generators
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Use the GPU when available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

# Shared plot style: colour-blind-safe palette, light grid
PALETTE = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#4a3aa7', '#e34948']
sns.set_theme(style='whitegrid', palette=PALETTE, font_scale=1.2,
              rc={'grid.color': '#e1e0d9', 'axes.edgecolor': '#c3c2b7', 'lines.linewidth': 2})

### ⏳ **Data Loading**

In [ ]:
# Load the Iris dataset and print its description
data = load_iris()
print(data.DESCR)

<img src="https://miro.medium.com/v2/resize:fit:1400/1*f6KbPXwksAliMIsibFyGJw.png" width="800">

### 🔎 **Exploration and Data Analysis**

In [ ]:
# Wrap the features in a DataFrame for exploration
iris_dataset = pd.DataFrame(data.data, columns=data.feature_names)
print(f"Iris dataset shape: {iris_dataset.shape}")
iris_dataset.head(10)

In [ ]:
# Summary statistics of each feature
iris_dataset.describe()

In [ ]:
# Class labels, class names and number of samples per class
target = data.target
class_names = [name.capitalize() for name in data.target_names]
labels, counts = np.unique(target, return_counts=True)
for label, count in zip(labels, counts):
    print(f"Class {label} ({class_names[label]}): {count} samples")

In [ ]:
# Pairwise feature relationships, coloured by species
plot_dataset = iris_dataset.assign(Species=[class_names[t] for t in target])
sns.pairplot(plot_dataset, hue='Species', palette=PALETTE[:3], markers=['o', 's', 'D'])
plt.show()

In [ ]:
# Input and output sizes of the network
input_features = iris_dataset.shape[1]
num_classes = len(class_names)
print(f"Number of input features: {input_features}")
print(f"Number of classes: {num_classes}")

### 🔄 **Data Preprocessing**

In [ ]:
# Features as float32 and labels as int64, the types PyTorch expects
X = iris_dataset.values.astype(np.float32)
y = target.astype(np.int64)

# First split: hold out 20 samples for the final test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=20, random_state=SEED, stratify=y
)

# Second split: 20 of the remaining samples for validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=20, random_state=SEED, stratify=y_train_val
)

print(f"Training set:   {X_train.shape}, {y_train.shape}")
print(f"Validation set: {X_val.shape}, {y_val.shape}")
print(f"Test set:       {X_test.shape}, {y_test.shape}")

In [ ]:
# Min-max scaling with statistics from the training set only, so no test information leaks in
min_df = X_train.min(axis=0)
max_df = X_train.max(axis=0)
print(f"Training minimum values: {min_df}")
print(f"Training maximum values: {max_df}")

X_train = (X_train - min_df) / (max_df - min_df)
X_val = (X_val - min_df) / (max_df - min_df)
X_test = (X_test - min_df) / (max_df - min_df)
print(f"Scaled training range: [{X_train.min():.1f}, {X_train.max():.1f}]")

In [ ]:
# Pair features with labels in PyTorch datasets
train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
val_ds = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
test_ds = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))

# Number of samples processed before each weight update
BATCH_SIZE = 16

# Shuffle only the training data; validation and test keep a fixed order
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

In [ ]:
# Inspect one training batch
xb, yb = next(iter(train_loader))
print(f"Features batch shape: {xb.shape}")
print(f"Labels batch shape: {yb.shape}")

### 🛠️ **Model Building**

In [ ]:
class FeedForwardNet(nn.Module):
    """Fully connected network with one hidden layer: input -> hidden (ReLU) -> output."""

    def __init__(self, in_features, hidden_size, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
# Number of neurons in the hidden layer
HIDDEN_SIZE = 32

model = FeedForwardNet(input_features, HIDDEN_SIZE, num_classes).to(device)
summary(model, input_size=(BATCH_SIZE, input_features))

The computational graph shows how a batch flows through the network: each box is a layer, each arrow a tensor with its shape.

In [ ]:
draw_graph(model, input_size=(BATCH_SIZE, input_features), expand_nested=True).visual_graph

### 🧮 **Hyperparameters**

In [ ]:
LEARNING_RATE = 1e-3
EPOCHS = 500

# Cross-entropy loss for multi-class classification, Adam optimiser
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

### 🧠 **Model Training**

An epoch is one full pass over the training set. Two functions do the work: `train_one_epoch` updates the weights batch by batch, `evaluate` measures loss and accuracy on a dataset without updating anything.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """Run one training epoch; return mean loss and accuracy."""
    model.train()
    total_loss, targets, preds = 0.0, [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        # Forward pass: predictions and loss
        logits = model(x)
        loss = criterion(logits, y)

        # Backward pass: gradients, then weight update
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        targets.append(y.cpu())
        preds.append(logits.argmax(dim=1).cpu())

    return total_loss / len(loader.dataset), accuracy_score(torch.cat(targets), torch.cat(preds))

In [ ]:
def evaluate(model, loader, criterion, device):
    """Return mean loss and accuracy on a loader, without updating the weights."""
    model.eval()
    total_loss, targets, preds = 0.0, [], []

    # No gradients are needed for evaluation
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            total_loss += criterion(logits, y).item() * x.size(0)
            targets.append(y.cpu())
            preds.append(logits.argmax(dim=1).cpu())

    return total_loss / len(loader.dataset), accuracy_score(torch.cat(targets), torch.cat(preds))

#### **Monitoring the Training**

`fit` repeats the two steps for a number of epochs and records the history of loss and accuracy. Its options switch each kind of monitoring on or off:
- `verbose=N` prints a log line every N epochs (`0` = silent);
- `live=N` redraws the training curves every N epochs while the network trains (`0` = off);
- `boundary` adds the validation decision boundary next to the live curves (`None` = off);
- `track_parameters=True` records weights, biases and their gradients after every epoch, for the histograms further down.

In [ ]:
def plot_history(history, boundary=None, max_epochs=None):
    """Plot training and validation curves of the loss and of each metric, plus the latest decision boundary if given."""
    metrics = [key.removeprefix('train_') for key in history if key.startswith('train_')]
    n_panels = len(metrics) + (boundary is not None)
    fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 4.5))
    epochs = range(1, len(history['train_loss']) + 1)

    for ax, metric in zip(axes, metrics):
        ax.plot(epochs, history[f'train_{metric}'], color=PALETTE[0], label='Training')
        ax.plot(epochs, history[f'val_{metric}'], color=PALETTE[1], label='Validation')
        ax.set(title=metric.capitalize(), xlabel='Epoch', xlim=(1, max_epochs or len(epochs)))
        ax.legend()

    if boundary is not None:
        boundary.plot(axes[-1], title='Decision boundary (validation)')
    plt.tight_layout()
    plt.show()

The network sees 4 features, so its decision regions live in a 4D space. To draw them we take the plane of the first two **principal components** of the training set, which keeps almost all of its variance, and ask the network to classify every point of a grid on that plane. `DecisionBoundary2D` holds this view: `update` classifies the grid with the current model, `plot` draws the regions with the validation samples on top.

In [ ]:
class DecisionBoundary2D:
    """Decision regions of a classifier, drawn on the plane of the first two principal components."""

    def __init__(self, X_train, X_val, y_val, class_names, resolution=200):
        # Plane of the first two principal components of the training data
        self.pca = PCA(n_components=2).fit(X_train)
        train_2d = self.pca.transform(X_train)

        # Grid of points on that plane, mapped back to the input features
        low, high = train_2d.min(axis=0) - 0.3, train_2d.max(axis=0) + 0.3
        self.grid_x, self.grid_y = np.meshgrid(np.linspace(low[0], high[0], resolution),
                                               np.linspace(low[1], high[1], resolution))
        grid = self.pca.inverse_transform(np.c_[self.grid_x.ravel(), self.grid_y.ravel()])
        self.grid_inputs = torch.from_numpy(grid.astype(np.float32))

        # Validation samples drawn on top of the regions
        self.points, self.labels = self.pca.transform(X_val), y_val
        self.class_names = class_names
        self.regions = None

    def update(self, model):
        """Classify every grid point with the current model."""
        model.eval()
        with torch.no_grad():
            preds = model(self.grid_inputs.to(next(model.parameters()).device)).argmax(dim=1)
        self.regions = preds.cpu().numpy().reshape(self.grid_x.shape)

    def plot(self, ax, title=None):
        """Draw the latest decision regions and the validation samples."""
        n = len(self.class_names)
        ax.contourf(self.grid_x, self.grid_y, self.regions, levels=np.arange(n + 1) - 0.5,
                    colors=PALETTE[:n], alpha=0.2)
        for c, name in enumerate(self.class_names):
            ax.scatter(*self.points[self.labels == c].T, color=PALETTE[c], s=30,
                       edgecolor='white', linewidth=0.5, label=name)
        ax.set(title=title, xlabel='PC 1', ylabel='PC 2')
        ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1))

In [ ]:
def record_parameters(model, record):
    """Append the current values and gradients of each linear layer's weights and bias."""
    linear_layers = [module for module in model.modules() if isinstance(module, nn.Linear)]
    for i, layer in enumerate(linear_layers, start=1):
        for kind, param in (('weights', layer.weight), ('bias', layer.bias)):
            entry = record.setdefault(f'Layer {i} {kind}', {'values': [], 'gradients': []})
            entry['values'].append(param.detach().cpu().flatten().numpy().copy())
            entry['gradients'].append(param.grad.detach().cpu().flatten().numpy().copy())

In [ ]:
def fit(model, train_loader, val_loader, criterion, optimizer, device, epochs,
        verbose=10, live=0, boundary=None, track_parameters=False):
    """Train for `epochs` epochs and return the history of loss, accuracy and, optionally, parameters."""
    history = {'train_loss': [], 'val_loss': [], 'train_accuracy': [], 'val_accuracy': []}
    if track_parameters:
        history['parameters'] = {}
    log = []

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        for key, value in zip(['train_loss', 'val_loss', 'train_accuracy', 'val_accuracy'],
                              [train_loss, val_loss, train_acc, val_acc]):
            history[key].append(value)
        if track_parameters:
            record_parameters(model, history['parameters'])

        # Which of the log line and the live plots are due at this epoch
        line = (f"Epoch {epoch:3d}/{epochs} | Train: loss {train_loss:.4f}, accuracy {train_acc:.4f} | "
                f"Val: loss {val_loss:.4f}, accuracy {val_acc:.4f}")
        log_now = verbose > 0 and (epoch == 1 or epoch % verbose == 0 or epoch == epochs)
        live_now = live > 0 and (epoch == 1 or epoch % live == 0 or epoch == epochs)

        if boundary is not None and live_now:
            boundary.update(model)
        if log_now:
            log.append(line)
            if not live:
                print(line)

        # Redraw the plots in place, with the log below ending on the current epoch
        if live_now:
            clear_output(wait=True)
            plot_history(history, boundary=boundary, max_epochs=epochs)
            print('\n'.join(log if log_now else log + [line]))

    return history

In [ ]:
# 2D view for the live decision boundary
boundary = DecisionBoundary2D(X_train, X_val, y_val, class_names)
print(f"Variance kept by the 2D view: {boundary.pca.explained_variance_ratio_.sum():.1%}")

In [ ]:
%%time
history = fit(model, train_loader, val_loader, criterion, optimizer, device, EPOCHS,
              verbose=50,               # log line every 50 epochs (0 = off)
              live=10,                  # redraw the plots every 10 epochs (0 = off)
              boundary=boundary,        # validation decision boundary in the live plots (None = off)
              track_parameters=True)    # weights, biases and gradients for the histograms below

# Save the trained weights to disk
torch.save(model.state_dict(), weights_path('model'))
print(f"Model saved: '{weights_path('model')}'")

### 🔬 **Inside the Training**

#### **Weights and Biases**

The same view as the histograms of TensorBoard. Each panel stacks the distribution of one tensor at successive epochs: the oldest at the back (top, light), the newest at the front (bottom, dark).

Watch the bias of the output layer (Layer 2 bias): it holds just three values, one per species, so each of them can be followed from epoch to epoch.

In [ ]:
def plot_histograms(parameters, kind='values', every=20):
    """TensorBoard-style histograms of each tensor, one slice every `every` epochs, the oldest at the back."""
    ramp = ['#cde2fb', '#0d366b'] if kind == 'values' else ['#fbd6c4', '#7a2a0a']
    colours = sns.blend_palette(ramp, as_cmap=True)
    kernel = np.exp(-0.5 * np.linspace(-2, 2, 9) ** 2)
    fig, axes = plt.subplots(1, len(parameters), figsize=(4.5 * len(parameters), 5))

    for ax, (name, record) in zip(axes, parameters.items()):
        data = record[kind]
        if kind == 'gradients':
            # Exact zeros (neurons inactive on the batch) would dwarf the rest: leave them out
            data = [values[values != 0] for values in data]
        epochs = np.r_[1, np.arange(every, len(data) + 1, every)]
        pooled = np.concatenate([data[e - 1] for e in epochs])

        # Values: the full range; gradients: the central 98%, so that a few outliers do not squeeze the plot.
        # The range is widened by 15% on each side so the smoothed tails are not clipped
        low, high = (pooled.min(), pooled.max()) if kind == 'values' else np.percentile(pooled, [1, 99])
        margin = 0.15 * (high - low) + 1e-8
        bins = np.linspace(low - margin, high + margin, 81)
        centres = (bins[:-1] + bins[1:]) / 2

        # Histogram of each slice, smoothed with a small Gaussian into a curve
        counts = np.stack([np.convolve(np.histogram(data[e - 1], bins)[0], kernel, mode='same')
                           for e in epochs])
        heights = 5 * counts / max(counts.max(), 1)

        # One ridge per slice, from the oldest (top, back) to the newest (bottom, front)
        for k, height in enumerate(heights):
            base = len(epochs) - 1 - k
            colour = colours(k / (len(epochs) - 1))
            ax.fill_between(centres, base, base + height, facecolor=colour,
                            edgecolor='white', linewidth=0.6, zorder=k)

        ticks = [k for k, e in enumerate(epochs) if k == 0 or e % (5 * every) == 0]
        ax.set_yticks([len(epochs) - 1 - k for k in ticks], [epochs[k] for k in ticks])
        ax.yaxis.tick_right()
        ax.yaxis.set_label_position('right')
        ax.set(title=name, xlabel='Value' if kind == 'values' else 'Gradient', ylabel='Epoch',
               xlim=(bins[0], bins[-1]))
        # A few ticks in scientific notation, so that small gradients stay readable
        ax.xaxis.set_major_locator(plt.MaxNLocator(5))
        ax.ticklabel_format(axis='x', style='sci', scilimits=(-2, 3))
        ax.grid(False)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_histograms(history['parameters'], kind='values')

#### **Gradients**

The gradients of the same tensors, taken after the last batch of each epoch. Their width tells how large the updates are: wide gradients mean large steps, narrow ones small steps. Gradients that collapse to zero or explode are the first sign of a network that has stopped learning or become unstable. Exact zeros, from neurons inactive on that batch, are left out, and the axis covers the central 98% of the values, so that the shape of the distribution stays visible.

In [ ]:
plot_histograms(history['parameters'], kind='gradients')

### 🕹️ **Inference**

In [ ]:
# Reload the trained weights from disk into a fresh model
best_model = FeedForwardNet(input_features, HIDDEN_SIZE, num_classes).to(device)
best_model.load_state_dict(torch.load(weights_path('model'), map_location=device))
best_model.eval()

In [ ]:
def predict(model, loader, device):
    """Return true and predicted labels for every sample in a loader."""
    model.eval()
    targets, preds = [], []
    with torch.no_grad():
        for x, y in loader:
            preds.append(model(x.to(device)).argmax(dim=1).cpu())
            targets.append(y)
    return torch.cat(targets).numpy(), torch.cat(preds).numpy()


def print_metrics(y_true, y_pred):
    """Print accuracy and the precision, recall and F1 score averaged over the classes."""
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, average='macro'):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred, average='macro'):.4f}")
    print(f"F1 score:  {f1_score(y_true, y_pred, average='macro'):.4f}")


def plot_confusion_matrix(y_true, y_pred, class_names, normalize=False, title='Confusion matrix'):
    """Plot a confusion matrix: rows are true classes, columns predicted classes."""
    cm = confusion_matrix(y_true, y_pred, normalize='true' if normalize else None)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='.2f' if normalize else 'd', cmap='Blues', cbar=False,
                square=True, xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set(xlabel='Predicted label', ylabel='True label', title=title)
    ax.tick_params(axis='y', rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
y_true, y_pred = predict(best_model, test_loader, device)
print_metrics(y_true, y_pred)
plot_confusion_matrix(y_true, y_pred, class_names, title='Confusion matrix (test set)')

### 📜 **Homework Exercise**
Perform the same classification analysis on the Penguins dataset to predict the correct species. Pay attention to missing values!

<img src="https://allisonhorst.github.io/palmerpenguins/reference/figures/lter_penguins.png" width="800">

```
# Data loading
DATASET_NAME = "penguins.csv"
DATASET_ID = "1qn1P6_KW08wGRfSkTlzBoDCyVb18T3Lk"
if not os.path.exists(DATASET_NAME):
    !gdown -q {DATASET_ID} -O {DATASET_NAME}
dataset = pd.read_csv(DATASET_NAME)
```

#  
<img src="https://airlab.deib.polimi.it/wp-content/uploads/2019/07/airlab-logo-new_cropped.png" width="350">

##### Connect with us:
- <img src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" width="14"> **LinkedIn:**  [AIRLab Polimi](https://www.linkedin.com/company/airlab-polimi/)
- <img src="https://upload.wikimedia.org/wikipedia/commons/9/95/Instagram_logo_2022.svg" width="14"> **Instagram:** [airlab_polimi](https://www.instagram.com/airlab_polimi/)

##### Contributors:
- **Eugenio Lomurno**: eugenio.lomurno@polimi.it
- **Alberto Archetti**: alberto.archetti@polimi.it
- **Roberto Basla**: roberto.basla@polimi.it
- **Carlo Sgaravatti**: carlo.sgaravatti@polimi.it

```
   Copyright 2026 Eugenio Lomurno, Alberto Archetti, Roberto Basla, Carlo Sgaravatti

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.
```